# Compositional Image Retrieval on CelebA
## Final Project Submission: Baseline and Attribute-Space Weighted BCE Loss (Strategy 7)

This notebook implements and compares three retrieval methods on the CelebA dataset:
1. **Baseline 1: Simple Vector Addition (CLIP Space)**
2. **Baseline 2: Geodesic Delta Embedding (GDE) in Tangent Space**
3. **Our Method: Global Weighted BCE Loss Retrieval (Strategy 7) with a Raw CLIP MLP Classifier**

This notebook is self-contained and formatted as an academic report with mathematical derivations and explanations.


## 1. Dataset and Environment Setup

We mount Google Drive to access the dataset zip (`celeba.zip`), evaluation metadata (`celeba_evaluation.json`), and precomputed CLIP embeddings (if available).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/datasets
# Unzip CelebA dataset from Google Drive to local SSD for fast I/O
!unzip -q /content/drive/MyDrive/datasets/celeba.zip -d /content/datasets/


### Load CelebA dataset and dependencies


In [ ]:
import os
import re
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from torchvision.datasets import CelebA
from transformers import CLIPProcessor, CLIPModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
data_root = Path('/content/datasets')
celeba = CelebA(root=data_root, split='test', download=False)
print(f'Dataset loaded. Total test images: {len(celeba)} using device: {device}')


## 2. Model Loading and Embedding Setup

We load OpenAI's CLIP model (`clip-vit-base-patch32`) from Hugging Face. We will check if pre-computed embeddings exist in Google Drive to avoid redundant CPU/GPU extraction. If they don't exist, we will extract them on the fly.


In [ ]:
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)

def embed_images_batched(dataset, model, processor, batch_size=256, device='cuda'):
    all_embeddings = []
    def custom_collate_fn(batch):
        images_pil = [item[0] for item in batch]
        inputs = processor(images=images_pil, return_tensors='pt', padding=True)
        return inputs

    data_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        collate_fn=custom_collate_fn
    )
    model.eval()
    with torch.no_grad():
        for batch_inputs in tqdm(data_loader, desc='Embedding images'):
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            image_features = model.get_image_features(**batch_inputs)
            all_embeddings.append(image_features.pooler_output.cpu())
    return torch.cat(all_embeddings)

drive_embeddings_path = Path('/content/drive/MyDrive/datasets/celeba_test_embeddings.pt')
if drive_embeddings_path.exists():
    print('Loading precomputed test embeddings from Google Drive...')
    E_test = torch.load(drive_embeddings_path, map_location=device)
else:
    print('Precomputed embeddings not found. Extracting embeddings (takes ~2 minutes on GPU)...')
    E_test = embed_images_batched(celeba, model, processor, batch_size=256, device=device).to(device)

E_test = F.normalize(E_test, p=2, dim=-1)
print(f'CLIP Test Embeddings Shape: {E_test.shape}')


## 3. Evaluation Metrics & Utility Functions

Here we load the search annotations and define query parsing, evaluation metrics, and print utilities. We calculate Recall@K and Precision@K, and define F1-Score as:
$$F_1@K = 2 \times \frac{\text{Precision}@K \times \text{Recall}@K}{\text{Precision}@K + \text{Recall}@K}$$
Our final comparison is based on the **Composite F1-Score**, which is the average of $F_1@1$, $F_1@5$, and $F_1@10$:


In [ ]:
annotations_path = Path('/content/drive/MyDrive/datasets/celeba_evaluation.json')
with open(annotations_path, 'r') as f:
    annotations = json.load(f)
print(f'Loaded {len(annotations)} queries.')

def parse_query(query_str):
    pos_attrs, neg_attrs = [], []
    parts = re.split(r'[&,]', query_str)
    for part in parts:
        part = part.strip()
        if part.startswith('+'):
            pos_attrs.append(part[1:].strip())
        elif part.startswith('-'):
            neg_attrs.append(part[1:].strip())
    return pos_attrs, neg_attrs

def evaluate_retrieval(retrieved_indices, ground_truth_indices, k):
    top_k_retrieved = retrieved_indices[:k]
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)
    recall_at_k = 1 if num_hits > 0 else 0
    precision_at_k = num_hits / k
    return {
        f'Recall@{k}': recall_at_k,
        f'Precision@{k}': precision_at_k
    }

def calculate_composite_f1(p1, r1, p5, r5, p10, r10):
    def calc_f1(p, r):
        if p + r == 0: return 0.0
        return 2.0 * p * r / (p + r)
    f1_1 = calc_f1(p1, r1)
    f1_5 = calc_f1(p5, r5)
    f1_10 = calc_f1(p10, r10)
    return (f1_1 + f1_5 + f1_10) / 3.0


## 4. Baseline 1: Simple Vector Addition (CLIP Space)

### Mathematical Explanation
Under this baseline, we assume that the CLIP text prompt directly represents the modification query. For a query $q$ (e.g. `+Smiling`), we first compose a natural language prompt $p_q$ (e.g. `"a photo of a person with smiling"`), and encode it to get a normalized text embedding $t_q \in \mathbb{S}^{D-1}$.

We then combine the source image embedding $u_{src} \in \mathbb{S}^{D-1}$ and the text embedding $t_q$ by taking their average:
$$u_{target} = \text{normalize}\left( \frac{u_{src} + t_q}{2}, p=2 \right)$$
The similarity score for candidate $c$ is computed using the cosine similarity (dot product):
$$\text{Score}(c) = u_{target}^T x_c$$
The candidates are sorted in descending order of similarity, and the Top-K are retrieved.


In [ ]:
def compose_prompt(query_str):
    pos_attrs, neg_attrs = parse_query(query_str)
    parts = []
    if pos_attrs:
        pos_part = ', '.join(attr.lower().replace('_', ' ') for attr in pos_attrs)
        parts.append(pos_part)
    if neg_attrs:
        neg_part = ', '.join(f'not {attr.lower().replace("_", " ")}' for attr in neg_attrs)
        parts.append(neg_part)
    combined = ', '.join(parts)
    return f'a photo of a person with {combined}'

def run_baseline1_evaluation(annotations, E_test, model, processor, device, cases_per_query=500):
    all_metrics = []
    for query_data in annotations:
        query_str = query_data['query']
        ground_truth = query_data['ground_truth']
        
        # 1. Encode text prompt
        prompt = compose_prompt(query_str)
        inputs = processor(text=[prompt], return_tensors='pt', padding=True, truncation=True).to(device)
        with torch.no_grad():
            text_feature = model.get_text_features(**inputs)
            text_feature = F.normalize(text_feature, p=2, dim=-1).squeeze(0)
            
        query_results = []
        source_count = 0
        for src_img_str, gt_indices in ground_truth.items():
            if source_count >= cases_per_query: break
            source_idx = int(src_img_str)
            
            # 2. Combine source embedding and text embedding
            u_src = E_test[source_idx]
            u_target = F.normalize((u_src + text_feature) / 2.0, p=2, dim=-1)
            
            # 3. Retrieve Top-10
            scores = (E_test @ u_target.unsqueeze(-1)).squeeze(-1)
            _, top_k = torch.topk(scores, k=10)
            predictions = top_k.tolist()
            
            metrics = {}
            for k in [1, 5, 10]:
                res = evaluate_retrieval(predictions, gt_indices, k=k)
                metrics[f'Recall@{k}'] = res[f'Recall@{k}']
                metrics[f'Precision@{k}'] = res[f'Precision@{k}']
            metrics['gt_count'] = len(gt_indices)
            query_results.append(metrics)
            source_count += 1
            
        # Average query metrics
        avg_m = {}
        for k in [1, 5, 10]:
            avg_m[f'Recall@{k}'] = sum(r[f'Recall@{k}'] for r in query_results) / len(query_results)
            avg_m[f'Precision@{k}'] = sum(r[f'Precision@{k}'] for r in query_results) / len(query_results)
        all_metrics.append((query_str, avg_m))
        
    # Print Summary
    print('=== BASELINE 1: SIMPLE VECTOR ADDITION ===')
    r1 = sum(m[1]['Recall@1'] for m in all_metrics) / len(all_metrics)
    r5 = sum(m[1]['Recall@5'] for m in all_metrics) / len(all_metrics)
    r10 = sum(m[1]['Recall@10'] for m in all_metrics) / len(all_metrics)
    p1 = sum(m[1]['Precision@1'] for m in all_metrics) / len(all_metrics)
    p5 = sum(m[1]['Precision@5'] for m in all_metrics) / len(all_metrics)
    p10 = sum(m[1]['Precision@10'] for m in all_metrics) / len(all_metrics)
    comp = calculate_composite_f1(p1, r1, p5, r5, p10, r10)
    print(f'Recall@1: {r1:.4f} | Recall@5: {r5:.4f} | Recall@10: {r10:.4f}')
    print(f'Precision@1: {p1:.4f} | Precision@5: {p5:.4f} | Precision@10: {p10:.4f}')
    print(f'Composite F1-Score: {comp:.4f}')
    return r1, r5, r10, p1, p5, p10, comp

r1_b1, r5_b1, r10_b1, p1_b1, p5_b1, p10_b1, comp_b1 = run_baseline1_evaluation(annotations, E_test, model, processor, device)


## 5. Baseline 2: Geodesic Delta Embedding (GDE) in Tangent Space

### Mathematical Explanation
Because CLIP embeddings lie on a hypersphere $\mathbb{S}^{D-1}$, GDE projects vectors onto the tangent space $T_{u_{src}}\mathbb{S}^{D-1}$ to prevent geometric distortion during translation. 

1. **Logarithmic Map:** Maps a sphere coordinate $x \in \mathbb{S}^{D-1}$ to the tangent space of $u_{src}$:
   $$v = \text{log}_{u_{src}}(x) = \frac{\theta}{\sin \theta} (x - \cos \theta \cdot u_{src})$$
   where $\cos \theta = u_{src}^T x$ and $\theta = \arccos(u_{src}^T x)$.
2. **Tangent Query Construction:** For each query, we synthesize a target direction $v_{query}$ using pre-computed primitive vectors $d_a = \mu^+_a - \mu^-_a$ in the tangent space:
   $$v_{query} = \sum_{a \in +Attr} d_a - \sum_{a \in -Attr} d_a$$
3. **Exponential Map:** Maps $v_{query}$ back onto the sphere:
   $$u_{target} = \text{exp}_{u_{src}}(v_{query}) = \cos(\|v_{query}\|) u_{src} + \sin(\|v_{query}\|) \frac{v_{query}}{\|v_{query}\|}$$
4. **Retrieval:** Score is calculated via $\text{Score}(c) = u_{target}^T x_c$.


In [ ]:
def log_map(u, x):
    # Projects x onto the tangent space of u
    cos_theta = torch.clamp(torch.dot(u, x), -1.0 + 1e-7, 1.0 - 1e-7)
    theta = torch.acos(cos_theta)
    if theta < 1e-6:
        return torch.zeros_like(x)
    return (theta / torch.sin(theta)) * (x - cos_theta * u)

def exp_map(u, v):
    # Maps tangent vector v back to the sphere
    norm_v = torch.norm(v)
    if norm_v < 1e-6:
        return u
    return torch.cos(norm_v) * u + torch.sin(norm_v) * (v / norm_v)

def run_baseline2_evaluation(annotations, E_test, celeba, device, cases_per_query=500):
    attr_names = [name for name in celeba.attr_names if name]
    name_to_idx = {name.lower().replace(' ', '_'): i for i, name in enumerate(attr_names)}
    Y_test = celeba.attr[:E_test.shape[0]].to(device).float()
    
    # Precompute attribute means in CLIP space to construct primitive directions
    # Positive and negative centroids
    centroids_pos = []
    centroids_neg = []
    for a in range(40):
        pos_mask = (Y_test[:, a] == 1.0)
        neg_mask = (Y_test[:, a] == 0.0) # In CelebA, negative is 0.0 or -1.0
        centroids_pos.append(E_test[pos_mask].mean(dim=0) if pos_mask.any() else torch.zeros(512, device=device))
        centroids_neg.append(E_test[neg_mask].mean(dim=0) if neg_mask.any() else torch.zeros(512, device=device))
    
    all_metrics = []
    for query_data in annotations:
        query_str = query_data['query']
        ground_truth = query_data['ground_truth']
        pos_attrs, neg_attrs = parse_query(query_str)
        pos_idx = [name_to_idx[a.strip().lower().replace(' ', '_')] for a in pos_attrs if a.strip().lower().replace(' ', '_') in name_to_idx]
        neg_idx = [name_to_idx[a.strip().lower().replace(' ', '_')] for a in neg_attrs if a.strip().lower().replace(' ', '_') in name_to_idx]
        
        query_results = []
        source_count = 0
        for src_img_str, gt_indices in ground_truth.items():
            if source_count >= cases_per_query: break
            source_idx = int(src_img_str)
            u_src = E_test[source_idx]
            
            # Compute primitive vectors in the tangent space of u_src
            v_query = torch.zeros(512, device=device)
            for a in pos_idx:
                d_pos = log_map(u_src, centroids_pos[a])
                d_neg = log_map(u_src, centroids_neg[a])
                v_query += (d_pos - d_neg)
            for a in neg_idx:
                d_pos = log_map(u_src, centroids_pos[a])
                d_neg = log_map(u_src, centroids_neg[a])
                v_query -= (d_pos - d_neg)
                
            # Map query direction back to sphere
            u_target = exp_map(u_src, v_query)
            u_target = F.normalize(u_target, p=2, dim=-1)
            
            # Retrieve Top-10
            scores = (E_test @ u_target.unsqueeze(-1)).squeeze(-1)
            _, top_k = torch.topk(scores, k=10)
            predictions = top_k.tolist()
            
            metrics = {}
            for k in [1, 5, 10]:
                res = evaluate_retrieval(predictions, gt_indices, k=k)
                metrics[f'Recall@{k}'] = res[f'Recall@{k}']
                metrics[f'Precision@{k}'] = res[f'Precision@{k}']
            metrics['gt_count'] = len(gt_indices)
            query_results.append(metrics)
            source_count += 1
            
        avg_m = {}
        for k in [1, 5, 10]:
            avg_m[f'Recall@{k}'] = sum(r[f'Recall@{k}'] for r in query_results) / len(query_results)
            avg_m[f'Precision@{k}'] = sum(r[f'Precision@{k}'] for r in query_results) / len(query_results)
        all_metrics.append((query_str, avg_m))
        
    print('=== BASELINE 2: GEODESIC DELTA EMBEDDING (GDE) ===')
    r1 = sum(m[1]['Recall@1'] for m in all_metrics) / len(all_metrics)
    r5 = sum(m[1]['Recall@5'] for m in all_metrics) / len(all_metrics)
    r10 = sum(m[1]['Recall@10'] for m in all_metrics) / len(all_metrics)
    p1 = sum(m[1]['Precision@1'] for m in all_metrics) / len(all_metrics)
    p5 = sum(m[1]['Precision@5'] for m in all_metrics) / len(all_metrics)
    p10 = sum(m[1]['Precision@10'] for m in all_metrics) / len(all_metrics)
    comp = calculate_composite_f1(p1, r1, p5, r5, p10, r10)
    print(f'Recall@1: {r1:.4f} | Recall@5: {r5:.4f} | Recall@10: {r10:.4f}')
    print(f'Precision@1: {p1:.4f} | Precision@5: {p5:.4f} | Precision@10: {p10:.4f}')
    print(f'Composite F1-Score: {comp:.4f}')
    return r1, r5, r10, p1, p5, p10, comp

r1_b2, r5_b2, r10_b2, p1_b2, p5_b2, p10_b2, comp_b2 = run_baseline2_evaluation(annotations, E_test, celeba, device)


## 6. Our Method: Global Weighted BCE Loss Retrieval (Strategy 7)

### Mathematical Explanation
GDE and simple vector addition suffer from primitive vector noise and visual identity drift. To fix this, our method trains an MLP classifier $f_\theta$ to map raw CLIP embeddings $x \in \mathbb{S}^{D-1}$ to attribute probability distributions $P_{c, a} = f_\theta(x)_a \in [0, 1]$ for all 40 CelebA attributes.

For any query and source index $s$, we build a continuous target profile $Y^*$:
$$Y^*_a = \begin{cases} 1.0 & \text{if } a \in +Attr \\ 0.0 & \text{if } a \in -Attr \\ P_{s, a} & \text{otherwise} \end{cases}$$
We then compute the weighted Binary Cross Entropy (BCE) loss globally against all candidates $c$ in the database:
$$\text{BCE Loss}(c) = - \frac{1}{\sum W_a} \sum_{a=1}^{40} W_a \cdot \left[ Y^*_a \ln P_{c, a} + (1 - Y^*_a) \ln (1 - P_{c, a}) \right]$$
where $W_a = \omega = 4.0$ for query-modified attributes, and $W_a = 1.0$ otherwise. Candidates with the **smallest** BCE loss are retrieved.

### Classifier Training Pipeline
We train the MLP classifier below using `BCELoss` on the training set embeddings (`celeba_train_embeddings.pt`) for 50 epochs.


In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim=512, hidden_dim1=256, hidden_dim2=128, output_dim=40):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.LayerNorm(hidden_dim1),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.LayerNorm(hidden_dim2),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(hidden_dim2, output_dim),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

def train_mlp_classifier(device, data_root):
    print('Loading training data...')
    E_train = torch.load(data_root / 'celeba_train_embeddings.pt', map_location=device)
    E_train = F.normalize(E_train, p=2, dim=-1)
    
    E_val = torch.load(data_root / 'celeba_val_embeddings.pt', map_location=device)
    E_val = F.normalize(E_val, p=2, dim=-1)
    
    celeba_train = CelebA(root=data_root, split='train', download=False)
    Y_train = celeba_train.attr[:E_train.shape[0]].to(device).float()
    
    celeba_val = CelebA(root=data_root, split='valid', download=False)
    Y_val = celeba_val.attr[:E_val.shape[0]].to(device).float()
    
    model = MLPClassifier(input_dim=512).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    
    epochs = 50
    batch_size = 4096
    num_samples = E_train.shape[0]
    best_val_loss = float('inf')
    best_state = None
    
    print('Training raw CLIP MLP classifier...')
    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(num_samples)
        epoch_loss = 0.0
        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            batch_x = E_train[indices]
            batch_y = Y_train[indices]
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * batch_x.shape[0]
            
        model.eval()
        with torch.no_grad():
            val_outputs = model(E_val)
            val_loss = criterion(val_outputs, Y_val).item()
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:02d}/{epochs} | Train Loss: {epoch_loss/num_samples:.4f} | Val Loss: {val_loss:.4f}')
            
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return model

drive_model_path = Path('/content/drive/MyDrive/datasets/gde_train_mlp_classifier_raw.pt')
mlp_model = MLPClassifier(input_dim=512).to(device)
if drive_model_path.exists():
    print('Loading trained model checkpoint from Google Drive...')
    mlp_model.load_state_dict(torch.load(drive_model_path, map_location=device))
else:
    mlp_model = train_mlp_classifier(device, data_root)
    torch.save(mlp_model.state_dict(), 'gde_train_mlp_classifier_raw.pt')
mlp_model.eval()


### Evaluate Strategy 7: Global Weighted BCE Loss (omega=4.0)


In [ ]:
def run_strategy7_evaluation(annotations, E_test, mlp_model, celeba, device, cases_per_query=500, omega=4.0):
    attr_names = [name for name in celeba.attr_names if name]
    name_to_idx = {name.lower().replace(' ', '_'): i for i, name in enumerate(attr_names)}
    
    # Precompute predictions of candidates globally
    print('Precomputing candidate attribute probabilities...')
    with torch.no_grad():
        P_candidates = []
        batch_size = 4096
        for i in range(0, E_test.shape[0], batch_size):
            P_candidates.append(mlp_model(E_test[i : i + batch_size]))
        P_candidates = torch.cat(P_candidates, dim=0)
    P_candidates_clamped = torch.clamp(P_candidates, min=1e-6, max=1.0 - 1e-6)
    
    all_metrics = []
    for query_data in annotations:
        query_str = query_data['query']
        ground_truth = query_data['ground_truth']
        pos_attrs, neg_attrs = parse_query(query_str)
        pos_idx = [name_to_idx[a.strip().lower().replace(' ', '_')] for a in pos_attrs if a.strip().lower().replace(' ', '_') in name_to_idx]
        neg_idx = [name_to_idx[a.strip().lower().replace(' ', '_')] for a in neg_attrs if a.strip().lower().replace(' ', '_') in name_to_idx]
        
        query_results = []
        source_count = 0
        for src_img_str, gt_indices in ground_truth.items():
            if source_count >= cases_per_query: break
            source_idx = int(src_img_str)
            
            # Retrieve source probability distribution
            probs = P_candidates_clamped[source_idx]
            
            # Construct target profile Y*
            Y_target = probs.clone()
            for a in pos_idx: Y_target[a] = 1.0
            for a in neg_idx: Y_target[a] = 0.0
            
            # Compute Weighted BCE Loss
            y = Y_target.unsqueeze(0)
            w = torch.ones(40, device=device)
            for a in pos_idx + neg_idx: w[a] = omega
            w = w.unsqueeze(0)
            
            term1 = y * torch.log(P_candidates_clamped)
            term2 = (1.0 - y) * torch.log(1.0 - P_candidates_clamped)
            weighted_bce = - torch.sum(w * (term1 + term2), dim=-1) / w.sum()
            
            _, top_k = torch.topk(weighted_bce, k=10, largest=False)
            predictions = top_k.tolist()
            
            metrics = {}
            for k in [1, 5, 10]:
                res = evaluate_retrieval(predictions, gt_indices, k=k)
                metrics[f'Recall@{k}'] = res[f'Recall@{k}']
                metrics[f'Precision@{k}'] = res[f'Precision@{k}']
            metrics['gt_count'] = len(gt_indices)
            query_results.append(metrics)
            source_count += 1
            
        avg_m = {}
        for k in [1, 5, 10]:
            avg_m[f'Recall@{k}'] = sum(r[f'Recall@{k}'] for r in query_results) / len(query_results)
            avg_m[f'Precision@{k}'] = sum(r[f'Precision@{k}'] for r in query_results) / len(query_results)
        all_metrics.append((query_str, avg_m))
        
    print('=== STRATEGY 7: GLOBAL WEIGHTED BCE LOSS (omega=4.0) ===')
    r1 = sum(m[1]['Recall@1'] for m in all_metrics) / len(all_metrics)
    r5 = sum(m[1]['Recall@5'] for m in all_metrics) / len(all_metrics)
    r10 = sum(m[1]['Recall@10'] for m in all_metrics) / len(all_metrics)
    p1 = sum(m[1]['Precision@1'] for m in all_metrics) / len(all_metrics)
    p5 = sum(m[1]['Precision@5'] for m in all_metrics) / len(all_metrics)
    p10 = sum(m[1]['Precision@10'] for m in all_metrics) / len(all_metrics)
    comp = calculate_composite_f1(p1, r1, p5, r5, p10, r10)
    print(f'Recall@1: {r1:.4f} | Recall@5: {r5:.4f} | Recall@10: {r10:.4f}')
    print(f'Precision@1: {p1:.4f} | Precision@5: {p5:.4f} | Precision@10: {p10:.4f}')
    print(f'Composite F1-Score: {comp:.4f}')
    return r1, r5, r10, p1, p5, p10, comp

r1_s7, r5_s7, r10_s7, p1_s7, p5_s7, p10_s7, comp_s7 = run_strategy7_evaluation(annotations, E_test, mlp_model, celeba, device)


## 7. Comparative Analysis and Final Leaderboard

We compare the three methods side-by-side on the full test split evaluation benchmark (cases=500).


In [ ]:
print('=' * 80)
print('FINAL RETRIEVAL COMPARISON LEADERBOARD (sorted by Composite F1)')
print('=' * 80)
print(f'{"Method":<40} | {"R@1":>6} | {"R@5":>6} | {"R@10":>6} | {"P@1":>6} | {"P@5":>6} | {"P@10":>6} | {"Comp F1":>9}')
print('-' * 80)
methods = [
    ('Baseline 1: Simple Vector Addition', (r1_b1, r5_b1, r10_b1, p1_b1, p5_b1, p10_b1), comp_b1),
    ('Baseline 2: Geodesic Delta Embedding (GDE)', (r1_b2, r5_b2, r10_b2, p1_b2, p5_b2, p10_b2), comp_b2),
    ('Strategy 7: Global Weighted BCE (omega=4.0)', (r1_s7, r5_s7, r10_s7, p1_s7, p5_s7, p10_s7), comp_s7)
]
methods.sort(key=lambda x: x[2], reverse=True)
for name, m, score in methods:
    print(f'{name:<40} | {m[0]:>6.4f} | {m[1]:>6.4f} | {m[2]:>6.4f} | {m[3]:>6.4f} | {m[4]:>6.4f} | {m[5]:>6.4f} | {score:>9.4f}')
print('=' * 80)
